In [1]:
print("hi")

hi


In [3]:
import os
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf

import torch

from transformers import (
    AutoFeatureExtractor,
    WavLMModel
)

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

from silero_vad import (
    load_silero_vad,
    get_speech_timestamps
)

a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [5]:
feature_extractor = AutoFeatureExtractor.from_pretrained(
    "microsoft/wavlm-base-plus"
)

wavlm = WavLMModel.from_pretrained(
    "microsoft/wavlm-base-plus"
)

wavlm = wavlm.to(device)

wavlm.eval()

a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at microsoft/wavlm-base-plus were not used when initializing WavLMModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing WavLMModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing WavLMModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of WavLMModel were not in

WavLMModel(
  (feature_extractor): WavLMFeatureEncoder(
    (conv_layers): ModuleList(
      (0): WavLMGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x WavLMNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x WavLMNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): WavLMFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): WavLMEncoder(
    (pos_conv_embed): WavLMPositionalConvEmbedding(
      (conv): Parametrized

In [6]:
csv_path = (
    "../Merch_Datasets/"
    "final_vietnamese_toxic_utterance_dataset.csv"
)

df = pd.read_csv(csv_path)

print(df.shape)

(26947, 8)


In [7]:
csv_dir = os.path.dirname(csv_path)


def build_full_audio_path(relative_path):

    relative_path = relative_path.replace(
        "./",
        ""
    )

    full_path = os.path.normpath(

        os.path.join(
            csv_dir,
            relative_path
        )

    )

    return full_path


df["audio_path"] = (
    df["audio_path"]
    .apply(build_full_audio_path)
)

In [8]:
def load_npz_audio(npz_path):

    data = np.load(npz_path)

    audio = data["audio"]

    if "sr" in data:
        sr = int(data["sr"])
    else:
        sr = 16000

    return audio.astype(np.float32), sr

In [9]:
toxic_df = df[
    df["toxicity"] == 1
].reset_index(drop=True)

print(len(toxic_df))

11802


In [10]:
@torch.no_grad()
def extract_layer3_embedding(
    audio,
    sr=16000
):

    inputs = feature_extractor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    )

    inputs = {

        k: v.to(device)

        for k, v in inputs.items()
    }

    outputs = wavlm(

        **inputs,

        output_hidden_states=True

    )

    layer3 = outputs.hidden_states[3]

    embedding = (

        layer3.mean(dim=1)
        .squeeze()
        .cpu()
        .numpy()

    )

    return embedding

In [11]:
toxic_embeddings = []

In [12]:
for idx, row in toxic_df.iterrows():

    waveform, sr = load_npz_audio(

        row["audio_path"]

    )

    emb = extract_layer3_embedding(
        waveform,
        sr
    )

    toxic_embeddings.append(emb)

    if idx % 100 == 0:
        print(idx)

a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800


In [13]:
toxic_embeddings = np.array(
    toxic_embeddings
)

print(
    toxic_embeddings.shape
)

(11802, 768)


In [14]:
kmeans = KMeans(

    n_clusters=20,

    random_state=42

)

kmeans.fit(
    toxic_embeddings
)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",20
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",42
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


# Toxic prototypes

In [15]:
prototypes = (
    kmeans.cluster_centers_
)

print(
    prototypes.shape
)

(20, 768)


In [16]:
vad_model = load_silero_vad()

In [17]:
def get_vad_segments(

    audio,
    sr=16000

):

    timestamps = get_speech_timestamps(

        torch.tensor(audio),

        vad_model,

        sampling_rate=sr

    )

    segments = []

    for x in timestamps:

        start_sec = (
            x["start"] / sr
        )

        end_sec = (
            x["end"] / sr
        )

        segments.append(

            (
                start_sec,
                end_sec
            )

        )

    return segments

In [18]:
def build_candidate_segments(

    speech_segments,

    min_sec=1,

    max_sec=14

):

    candidates = []

    n = len(speech_segments)

    for i in range(n):

        start = speech_segments[i][0]

        end = start

        for j in range(i, n):

            end = speech_segments[j][1]

            duration = end - start

            if duration < min_sec:
                continue

            if duration > max_sec:
                break

            candidates.append(
                (
                    start,
                    end
                )
            )

    return candidates

# Hàm tính IoU

In [27]:
def interval_iou(
    a_start,
    a_end,
    b_start,
    b_end
):

    intersection = max(
        0,
        min(a_end, b_end)
        -
        max(a_start, b_start)
    )

    union = (
        (a_end - a_start)
        +
        (b_end - b_start)
        -
        intersection
    )

    if union == 0:
        return 0

    return intersection / union

# NMS với IoU + minimum gap

In [28]:
def temporal_nms(
    results,
    iou_threshold=0.3,
    min_gap_sec=3
):

    # sort theo similarity giảm dần
    results = sorted(
        results,
        key=lambda x: x["similarity"],
        reverse=True
    )

    selected = []

    for cand in results:

        keep = True

        cur_start = cand["start_sec"]
        cur_end = cand["end_sec"]

        for prev in selected:

            prev_start = prev["start_sec"]
            prev_end = prev["end_sec"]

            # ===================
            # IoU
            # ===================
            iou = interval_iou(
                cur_start,
                cur_end,
                prev_start,
                prev_end
            )

            if iou > iou_threshold:
                keep = False
                break

            # ===================
            # minimum time gap
            # ===================
            gap = max(
                0,
                max(cur_start, prev_start)
                -
                min(cur_end, prev_end)
            )

            if gap < min_gap_sec:
                keep = False
                break

        if keep:
            selected.append(cand)

    return selected

# Cosine similarity

In [29]:
def retrieve_segments(

    audio,
    sr,
    prototypes,
    threshold=0.80,
    iou_threshold=0.3,
    min_gap_sec=3

):

    results = []

    speech_segments = get_vad_segments(
        audio,
        sr
    )

    candidate_segments = build_candidate_segments(
        speech_segments
    )

    for start_sec, end_sec in candidate_segments:

        start_sample = int(
            start_sec * sr
        )

        end_sample = int(
            end_sec * sr
        )

        segment = audio[
            start_sample:end_sample
        ]

        if len(segment) < sr:
            continue

        emb = extract_layer3_embedding(
            segment,
            sr
        )

        sims = cosine_similarity(

            emb.reshape(1, -1),

            prototypes

        )[0]

        score = sims.max()

        if score >= threshold:

            results.append(

                {

                    "start_sec":
                        start_sec,

                    "end_sec":
                        end_sec,

                    "duration":
                        end_sec - start_sec,

                    "similarity":
                        float(score),

                    "prototype":
                        int(sims.argmax())

                }

            )

    # ===================
    # Non-Maximum Suppression
    # ===================
    results = temporal_nms(

        results,

        iou_threshold=iou_threshold,

        min_gap_sec=min_gap_sec

    )

    return results

# Scan audio mới

In [30]:
audio_dir = Path(
    r"A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\another_test\Get_More\audio"
)

all_candidates = []

for file in audio_dir.glob("*"):

    print(file)

    waveform, sr = librosa.load(

        file,
        sr=16000

    )

    results = retrieve_segments(

        waveform,

        sr,

        prototypes,

        threshold=0.82,
        iou_threshold=0.3,
        min_gap_sec=5
    )

    for x in results:

        x["source_file"] = str(file)

    all_candidates.extend(results)

A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\another_test\Get_More\audio\5LeH0RHuTAE.wav


a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-pack

A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\another_test\Get_More\audio\6ga71BYa480.wav


a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-pack

A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\another_test\Get_More\audio\8y3otCkVgrk.wav


...

a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-pack

A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\another_test\Get_More\audio\NbvKJ0b7ppc.wav


a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-pack

A:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\another_test\Get_More\audio\SPEurN60134.wav


a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
a:\A _ Working\Researching\B - AIoT Lab VN\VITOSA\envi\Lib\site-pack

In [31]:
print(len(all_candidates))

506


In [32]:
candidate_df = pd.DataFrame(
    all_candidates
)

candidate_df = (
    candidate_df
    .sort_values(
        "similarity",
        ascending=False
    )
)

candidate_df.head()

,start_sec,end_sec,duration,similarity,prototype,source_file
109,890.018,896.478,6.460,0.943215,7,A:\A _ Working\Researching\B - AIoT Lab VN\VIT...
370,848.994,856.670,7.676,0.943179,7,A:\A _ Working\Researching\B - AIoT Lab VN\VIT...
204,651.586,661.278,9.692,0.938974,2,A:\A _ Working\Researching\B - AIoT Lab VN\VIT...
110,224.674,234.206,9.532,0.937837,17,A:\A _ Working\Researching\B - AIoT Lab VN\VIT...
0,3599.970,3613.374,13.404,0.931403,7,A:\A _ Working\Researching\B - AIoT Lab VN\VIT...


# Save segment

In [33]:
from pathlib import Path
from collections import defaultdict

save_dir = "./candidate_segments"

os.makedirs(
    save_dir,
    exist_ok=True
)

# đếm số segment của từng file source
counter = defaultdict(int)

segment_names = []

for idx, row in candidate_df.iterrows():

    source_path = row["source_file"]

    # lấy tên file không có đuôi
    stem = Path(source_path).stem

    # tăng bộ đếm
    counter[stem] += 1

    # tên file mới
    segment_name = (
        f"{stem}_{counter[stem]}.wav"
    )

    segment_names.append(segment_name)

    # load audio
    audio, sr = librosa.load(
        source_path,
        sr=16000
    )

    # cắt segment
    start_sample = int(
        row["start_sec"] * sr
    )

    end_sample = int(
        row["end_sec"] * sr
    )

    segment = audio[
        start_sample:end_sample
    ]

    save_path = os.path.join(
        save_dir,
        segment_name
    )

    sf.write(
        save_path,
        segment,
        sr
    )

candidate_df["segment_filename"] = segment_names


# Lưu CSV
candidate_df.to_csv(
    "candidate_segments.csv",
    index=False
)

# save mode

In [34]:
save_dir = "./wavlm_layer3"

# save model
wavlm.save_pretrained(
    save_dir
)

# save feature extractor
feature_extractor.save_pretrained(
    save_dir
)

['./wavlm_layer3\\preprocessor_config.json']

# Load model 

In [ ]:
from transformers import (
    AutoFeatureExtractor,
    WavLMModel
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

save_dir = "./wavlm_layer3"

feature_extractor = (
    AutoFeatureExtractor.from_pretrained(
        save_dir
    )
)

wavlm = (
    WavLMModel.from_pretrained(
        save_dir
    )
    .to(device)
)

wavlm.eval()

In [35]:
prototypes = kmeans.cluster_centers_

np.save(
    "toxic_prototypes.npy",
    prototypes
)

In [ ]:
# load
prototypes = np.load(
    "toxic_prototypes.npy"
)